## importing libraries and Loading dataset

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np



In [5]:

transport_file = Path(
    "/Users/rohankhanna/Documents/track3_agritech_dataset_files/track3_transport_logistics.csv"
)

df_transport = pd.read_csv(transport_file)
df_transport_clean = df_transport.copy(deep=True)

print("Rows:", len(df_transport_clean))
print("Columns:", df_transport_clean.columns.tolist())
print("Exact duplicates:", df_transport_clean.duplicated().sum())

display(df_transport_clean.head(10))

Rows: 10400
Columns: ['trip_id', 'mandi_id', 'destination_warehouse', 'departure_time', 'arrival_time', 'transit_hours', 'distance', 'distance_unit', 'vehicle_no', 'driver_id']
Exact duplicates: 400


,trip_id,mandi_id,destination_warehouse,departure_time,arrival_time,transit_hours,distance,distance_unit,vehicle_no,driver_id
0,TRP006374,MANDI050,WH-Central,2026-04-30T20:08:22,01/05/2026 05:26,9.3,305.714532,miles,NaN,DRV264
1,TRP008869,MANDI029,WH-West,04-10-2026 04:15 AM,10/04/2026 09:51,NaN,263.5,km,NaN,DRV540
2,TRP000674,MANDI024,WH-South,18/07/2026 16:34,07-18-2026 10:46 PM,6.2,276.5,km,UP 50 BC 6882,DRV722
3,TRP000036,MANDI-042,WH-Central,08-13-2026 06:21 AM,14/08/2026,19.7,1070.7,km,RJ-52-CD-3274,DRV103
4,TRP008288,mandi_019,WH-South,07-30-2026 10:51 PM,07-31-2026 03:57 PM,17.1,728.9,km,DL-73-DF-1458,NaN
5,TRP003456,038,Export-Terminal,07-25-2026 08:53 PM,NaN,7.3,417.0,km,UP97-BC-6135,DRV255
6,TRP005236,mandi041,WH-West,2026-07-05 07:34:30,05/07/2026 22:28,14.9,671.9,km,PB-99-BC-2796,DRV297
7,TRP007075,023,WH-West,19/06/2026,2026-06-19T22:01:06,6.4,362.6,km,pb 48-DF-3590,NaN
8,TRP002222,mandi_033,WH-North,02-May-2026 10:12:36,02/05/2026 22:12,12.0,549.5,km,DL55-BC-7556,DRV662
9,TRP004565,MANDI054,WH-West,05/09/2026 10:36,05/09/2026 20:00,9.4,424.8,km,PB 83 DF 2210,DRV201


## Inspecting missing ID's

In [6]:
inspection = df_transport_clean.replace(
    r"^\s*$", pd.NA, regex=True
)

display(
    pd.DataFrame({
        "Data type": df_transport_clean.dtypes.astype(str),
        "Missing count": inspection.isna().sum()
    })
)

,Data type,Missing count
trip_id,str,0
mandi_id,str,0
destination_warehouse,str,0
departure_time,str,0
arrival_time,str,1053
transit_hours,str,518
distance,str,0
distance_unit,str,1032
vehicle_no,str,1622
driver_id,str,1551


## Checking trip id's and removind exact duplicates

In [7]:
# Start from the original transport data.
duplicate_count = df_transport.duplicated().sum()

df_transport_clean = (
    df_transport
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

print("Exact duplicates removed:", duplicate_count)
print("Remaining rows:", len(df_transport_clean))

# Standardize trip IDs without overwriting the originals.
df_transport_clean["trip_id_clean"] = (
    df_transport_clean["trip_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

ids = df_transport_clean["trip_id_clean"]
repeated = ids.notna() & ids.duplicated(keep=False)

print("Missing trip IDs:", ids.isna().sum())
print("Rows with repeated trip IDs:", repeated.sum())

Exact duplicates removed: 400
Remaining rows: 10000
Missing trip IDs: 0
Rows with repeated trip IDs: 0


## Standardizing Mandi id's and validating them against the masters

In [8]:
ids = (
    df_transport_clean["mandi_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
    .str.replace(r"[\s_-]+", "", regex=True)
)

numbers = ids.str.extract(
    r"^(?:MANDI|M)?(\d{1,3})$",
    expand=False
)

df_transport_clean["mandi_id_clean"] = (
    "MANDI" + numbers.str.zfill(3)
)

# Load the master into this notebook.
df_master_clean = pd.read_csv(
    "/Users/rohankhanna/Documents/datathon/cleaned_data/"
    "mandi_master_cleaned.csv",
    dtype={"mandi_id": "string"},
    na_values=["NA"]
)

clean_ids = df_transport_clean["mandi_id_clean"]

df_transport_clean["mandi_id_status"] = np.select(
    [
        ids.isna().to_numpy(dtype=bool),
        clean_ids.isna().to_numpy(dtype=bool),
        (~clean_ids.isin(df_master_clean["mandi_id"])).to_numpy(dtype=bool)
    ],
    [
        "Missing source ID",
        "Unrecognized format",
        "Not found in master"
    ],
    default="Matched"
)

display(
    df_transport_clean["mandi_id_status"]
    .value_counts()
    .to_frame("row_count")
)

,row_count
mandi_id_status,
Matched,10000


## Checking names and distance units

In [9]:
for column in ["destination_warehouse", "distance_unit"]:
    print(f"\nValues in {column}:")
    display(
        df_transport_clean[column]
        .value_counts(dropna=False)
        .to_frame("row_count")
    )

print("\nSample distances:")
display(
    df_transport_clean[["distance", "distance_unit"]].head(20)
)

print("\nExamples with missing distance units:")
missing_unit = (
    df_transport_clean["distance_unit"]
    .astype("string")
    .fillna("")
    .str.strip()
    .eq("")
)

display(
    df_transport_clean.loc[
        missing_unit, ["distance", "distance_unit"]
    ].head(10)
)


Values in destination_warehouse:


,row_count
destination_warehouse,
WH-North,1710
WH-South,1699
WH-West,1681
WH-Central,1660
Export-Terminal,1640
WH-East,1610



Values in distance_unit:


,row_count
distance_unit,
km,7511
miles,1497
NaN,992



Sample distances:


,distance,distance_unit
0,305.714532,miles
1,263.5,km
2,276.5,km
3,1070.7,km
4,728.9,km
5,417.0,km
6,671.9,km
7,362.6,km
8,549.5,km
9,424.8,km



Examples with missing distance units:


,distance,distance_unit
16,316.5 KM,NaN
37,196.7 KM,NaN
40,257.7 KM,NaN
58,842.6 KM,NaN
62,364.3 KM,NaN
66,644.4 KM,NaN
69,794.7 KM,NaN
70,1021.5 KM,NaN
78,914.9 KM,NaN
85,217.4 KM,NaN


## Parsing distances and resolving there units

In [10]:
df_transport_clean["destination_warehouse_clean"] = (
    df_transport_clean["destination_warehouse"]
    .astype("string").str.strip()
)

distance_text = (
    df_transport_clean["distance"]
    .astype("string").str.strip().str.lower()
)

parts = distance_text.str.extract(
    r"^([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([a-z]+)?$"
)

df_transport_clean["distance_numeric"] = pd.to_numeric(
    parts[0], errors="coerce"
)

column_unit = (
    df_transport_clean["distance_unit"]
    .astype("string").str.strip().str.lower()
    .replace("", pd.NA)
)

embedded_unit = parts[1]

conflict = (
    column_unit.notna()
    & embedded_unit.notna()
    & column_unit.ne(embedded_unit)
)

df_transport_clean["distance_unit_clean"] = (
    column_unit.fillna(embedded_unit).mask(conflict)
)

print(
    "Unparseable distances:",
    df_transport_clean["distance_numeric"].isna().sum()
)
print("Conflicting units:", conflict.sum())

display(
    df_transport_clean["distance_unit_clean"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

Unparseable distances: 0
Conflicting units: 0


,row_count
distance_unit_clean,
km,8503
miles,1497


## Converting distances to kilometers and flag invalid values

In [11]:
distance = df_transport_clean["distance_numeric"]
unit = df_transport_clean["distance_unit_clean"]

df_transport_clean["distance_status"] = np.select(
    [
        distance.isna().to_numpy(dtype=bool),
        (~unit.isin(["km", "miles"])).to_numpy(dtype=bool),
        distance.le(0).fillna(False).to_numpy(dtype=bool)
    ],
    [
        "Unparseable distance",
        "Missing or unresolved unit",
        "Nonpositive distance: review"
    ],
    default="Valid"
)

df_transport_clean["distance_km"] = (
    distance * unit.map({"km": 1, "miles": 1.609344})
).where(df_transport_clean["distance_status"].eq("Valid"))

display(
    df_transport_clean["distance_status"]
    .value_counts()
    .to_frame("row_count")
)

display(
    df_transport_clean[
        ["distance", "distance_unit_clean", "distance_km"]
    ].head(10)
)

,row_count
distance_status,
Valid,10000


,distance,distance_unit_clean,distance_km
0,305.714532,miles,491.999848
1,263.5,km,263.5
2,276.5,km,276.5
3,1070.7,km,1070.7
4,728.9,km,728.9
5,417.0,km,417.0
6,671.9,km,671.9
7,362.6,km,362.6
8,549.5,km,549.5
9,424.8,km,424.8


## Inspecting departure and arrival timestamp formats

In [12]:
for column in ["departure_time", "arrival_time"]:
    text = (
        df_transport_clean[column]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    print(f"\n{column} — formats:")
    display(
        text.str.replace(r"\d", "D", regex=True)
        .value_counts(dropna=False)
        .to_frame("row_count")
    )

    print("Sample values:")
    display(text.dropna().drop_duplicates().head(10).to_frame())


departure_time — formats:


,row_count
departure_time,
DDDD-DD-DD DD:DD:DD,2514
DD/DD/DDDD DD:DD,2017
DDDD-DD-DDTDD:DD:DD,1534
DD/DD/DDDD,928
DD-DD-DDDD DD:DD AM,746
DD-DD-DDDD DD:DD PM,739
DD-Apr-DDDD DD:DD:DD,209
DD-May-DDDD DD:DD:DD,193
DD-Mar-DDDD DD:DD:DD,191


Sample values:


,departure_time
0,2026-04-30T20:08:22
1,04-10-2026 04:15 AM
2,18/07/2026 16:34
3,08-13-2026 06:21 AM
4,07-30-2026 10:51 PM
5,07-25-2026 08:53 PM
6,2026-07-05 07:34:30
7,19/06/2026
8,02-May-2026 10:12:36
9,05/09/2026 10:36



arrival_time — formats:


,row_count
arrival_time,
DDDD-DD-DD DD:DD:DD,2258
DD/DD/DDDD DD:DD,1767
DDDD-DD-DDTDD:DD:DD,1353
<NA>,1006
DD/DD/DDDD,881
DD-DD-DDDD DD:DD PM,689
DD-DD-DDDD DD:DD AM,656
DD-Jan-DDDD DD:DD:DD,188
DD-Aug-DDDD DD:DD:DD,187


Sample values:


,arrival_time
0,01/05/2026 05:26
1,10/04/2026 09:51
2,07-18-2026 10:46 PM
3,14/08/2026
4,07-31-2026 03:57 PM
6,05/07/2026 22:28
7,2026-06-19T22:01:06
8,02/05/2026 22:12
9,05/09/2026 20:00
10,02/05/2026 18:45


## Checking numeric date conventions

In [13]:
for column in ["departure_time", "arrival_time"]:
    text = df_transport_clean[column].astype("string").str.strip()

    print(f"\n{column}")

    for separator in ["/", "-"]:
        parts = text.str.extract(
            rf"^(\d{{2}})[{separator}](\d{{2}})[{separator}]\d{{4}}(?:\s|$)"
        )

        first = pd.to_numeric(parts[0], errors="coerce")
        second = pd.to_numeric(parts[1], errors="coerce")

        print(
            f"{separator}: "
            f"first number > 12 = {first.gt(12).sum()}, "
            f"second number > 12 = {second.gt(12).sum()}"
        )


departure_time
/: first number > 12 = 1669, second number > 12 = 0
-: first number > 12 = 0, second number > 12 = 869

arrival_time
/: first number > 12 = 1541, second number > 12 = 0
-: first number > 12 = 0, second number > 12 = 804


In [14]:
formats = {
    r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}": "%Y-%m-%d %H:%M:%S",
    r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}": "%Y-%m-%dT%H:%M:%S",
    r"\d{2}/\d{2}/\d{4} \d{2}:\d{2}": "%d/%m/%Y %H:%M",
    r"\d{2}/\d{2}/\d{4}": "%d/%m/%Y",
    r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} [AP]M": "%m-%d-%Y %I:%M %p",
    r"\d{2}-[A-Za-z]{3}-\d{4} \d{2}:\d{2}:\d{2}": "%d-%b-%Y %H:%M:%S"
}

for column in ["departure_time", "arrival_time"]:
    text = (
        df_transport_clean[column].astype("string")
        .str.strip().replace("", pd.NA)
    )

    parsed = pd.Series(pd.NaT, index=df_transport_clean.index)

    for pattern, date_format in formats.items():
        mask = text.str.fullmatch(pattern, na=False)
        parsed.loc[mask] = pd.to_datetime(
            text.loc[mask], format=date_format, errors="coerce"
        )

    date_only = text.str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False)

    # Preserve the calendar date, but exclude date-only values from timestamps.
    df_transport_clean[f"{column}_date"] = parsed.dt.normalize()
    df_transport_clean[f"{column}_clean"] = parsed.mask(date_only)

    df_transport_clean[f"{column}_status"] = np.select(
        [
            text.isna().to_numpy(dtype=bool),
            parsed.isna().to_numpy(dtype=bool),
            date_only.to_numpy(dtype=bool)
        ],
        ["Missing", "Invalid format or date", "Date only"],
        default="Date and time available"
    )

    # Preserve the date interpretation assumption separately.
    parts = text.str.extract(r"^(\d{2})[/-](\d{2})[/-]\d{4}(?:\s|$)")
    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    df_transport_clean[f"{column}_date_assumed"] = (
        first.between(1, 12)
        & second.between(1, 12)
        & first.ne(second)
    )

    print(f"\n{column}:")
    display(
        df_transport_clean[f"{column}_status"]
        .value_counts().to_frame("row_count")
    )


departure_time:


,row_count
departure_time_status,
Date and time available,9072
Date only,928



arrival_time:


,row_count
arrival_time_status,
Date and time available,8113
Missing,1006
Date only,881


## Calculating transit_hours where both full timestamps exists

In [15]:
# Calculate elapsed hours from full timestamps only.
df_transport_clean["transit_hours_calculated"] = (
    df_transport_clean["arrival_time_clean"]
    - df_transport_clean["departure_time_clean"]
).dt.total_seconds() / 3600

# Inspect the supplied duration without overwriting it.
reported_text = (
    df_transport_clean["transit_hours"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df_transport_clean["transit_hours_reported"] = pd.to_numeric(
    reported_text, errors="coerce"
)

calculated = df_transport_clean["transit_hours_calculated"]
reported = df_transport_clean["transit_hours_reported"]

print("Trips with calculated duration:", calculated.notna().sum())
print("Negative calculated durations:", calculated.lt(0).sum())
print("Zero calculated durations:", calculated.eq(0).sum())

print("\nMissing reported durations:", reported_text.isna().sum())
print(
    "Unparseable reported durations:",
    (reported_text.notna() & reported.isna()).sum()
)
print("Negative reported durations:", reported.lt(0).sum())
print("Zero reported durations:", reported.eq(0).sum())

# Inspect agreement where both durations are nonnegative.
comparable = calculated.ge(0) & reported.ge(0)

difference = (
    calculated.loc[comparable] - reported.loc[comparable]
).abs()

print("\nAbsolute difference in hours:")
display(difference.describe())

Trips with calculated duration: 7359
Negative calculated durations: 0
Zero calculated durations: 0

Missing reported durations: 502
Unparseable reported durations: 504
Negative reported durations: 539
Zero reported durations: 0

Absolute difference in hours:


count      6223.0
mean      0.00383
std       0.00522
min           0.0
25%           0.0
50%           0.0
75%        0.0075
max      0.016389
dtype: Float64

## Inspecting unparseable durations

In [16]:
unparseable = (
    reported_text.notna()
    & df_transport_clean["transit_hours_reported"].isna()
)

display(
    df_transport_clean.loc[
        unparseable, ["transit_hours"]
    ]
    .value_counts()
    .head(20)
    .to_frame("row_count")
)

,row_count
transit_hours,
23.7 hrs,7
23.0 hrs,6
4.7 hrs,5
2.1 hrs,5
23.6 hrs,5
18.2 hrs,5
20.9 hrs,5
3.1 hrs,5
18.4 hrs,5


## Extracting our values as they contain hrs

In [17]:
reported_text = (
    df_transport_clean["transit_hours"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df_transport_clean["transit_hours_reported"] = pd.to_numeric(
    reported_text.str.replace(
        r"(?i)\s*hrs\s*$", "", regex=True
    ),
    errors="coerce"
)

reported = df_transport_clean["transit_hours_reported"]

print(
    "Still unparseable:",
    (reported_text.notna() & reported.isna()).sum()
)
print("Missing:", reported_text.isna().sum())
print("Negative:", reported.lt(0).sum())

Still unparseable: 0
Missing: 502
Negative: 539


## Final transit duration

In [18]:
calculated = df_transport_clean["transit_hours_calculated"]
reported = df_transport_clean["transit_hours_reported"]

use_calculated = calculated.gt(0).fillna(False)

use_reported = (
    calculated.isna()
    & reported.gt(0)
).fillna(False)

df_transport_clean["transit_hours_clean"] = (
    calculated.where(use_calculated)
)

df_transport_clean.loc[
    use_reported, "transit_hours_clean"
] = reported.loc[use_reported]

df_transport_clean["transit_duration_source"] = np.select(
    [
        use_calculated.to_numpy(dtype=bool),
        use_reported.to_numpy(dtype=bool)
    ],
    [
        "Calculated from timestamps",
        "Reported: timestamps incomplete"
    ],
    default="Unavailable or invalid"
)

# Preserve the negative-source issue even if timestamps recover the duration.
df_transport_clean["reported_transit_negative"] = (
    reported.lt(0).fillna(False)
)

display(
    df_transport_clean["transit_duration_source"]
    .value_counts()
    .to_frame("row_count")
)

,row_count
transit_duration_source,
Calculated from timestamps,7359
Reported: timestamps incomplete,2356
Unavailable or invalid,285


## Final check

In [19]:
calculated = df_transport_clean["transit_hours_calculated"]
reported = df_transport_clean["transit_hours_reported"]

comparable = calculated.gt(0) & reported.gt(0)

df_transport_clean["transit_difference_hours"] = (
    calculated - reported
).abs().where(comparable)

display(
    df_transport_clean["transit_difference_hours"].describe()
)

count      6603.0
mean     0.003823
std      0.005212
min           0.0
25%           0.0
50%           0.0
75%        0.0075
max      0.016389
Name: transit_difference_hours, dtype: Float64

## Standardizing vehicles and driver id's

In [20]:
# Normalize vehicle labels, e.g. "pb 48-DF-3590" → "PB48DF3590".
df_transport_clean["vehicle_no_clean"] = (
    df_transport_clean["vehicle_no"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"[\s-]+", "", regex=True)
    .replace("", pd.NA)
)

df_transport_clean["driver_id_clean"] = (
    df_transport_clean["driver_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

display(
    df_transport_clean[
        ["vehicle_no", "vehicle_no_clean", "driver_id_clean"]
    ].head(15)
)

print(
    "Missing vehicle numbers:",
    df_transport_clean["vehicle_no_clean"].isna().sum()
)
print(
    "Missing driver IDs:",
    df_transport_clean["driver_id_clean"].isna().sum()
)

,vehicle_no,vehicle_no_clean,driver_id_clean
0,NaN,<NA>,DRV264
1,NaN,<NA>,DRV540
2,UP 50 BC 6882,UP50BC6882,DRV722
3,RJ-52-CD-3274,RJ52CD3274,DRV103
4,DL-73-DF-1458,DL73DF1458,<NA>
5,UP97-BC-6135,UP97BC6135,DRV255
6,PB-99-BC-2796,PB99BC2796,DRV297
7,pb 48-DF-3590,PB48DF3590,<NA>
8,DL55-BC-7556,DL55BC7556,DRV662
9,PB 83 DF 2210,PB83DF2210,DRV201


Missing vehicle numbers: 1560
Missing driver IDs: 1512


## checking unexpected formats

In [22]:
vehicles = df_transport_clean["vehicle_no_clean"]
drivers = df_transport_clean["driver_id_clean"]

vehicle_review = (
    vehicles.notna()
    & ~vehicles.str.fullmatch(
        r"[A-Z]{2}\d{2}[A-Z]{1,3}\d{4}", na=False
    )
)

driver_review = (
    drivers.notna()
    & ~drivers.str.fullmatch(r"DRV\d+", na=False)
)

print("Vehicle formats to review:", int(vehicle_review.sum()))
print("Driver formats to review:", int(driver_review.sum()))

Vehicle formats to review: 0
Driver formats to review: 0


## Final Validations

In [24]:
# Remove optional analytical columns if they were created.
df_transport_clean = df_transport_clean.drop(
    columns=["route_p75_hours", "long_transit_flag"],
    errors="ignore"
)

assert len(df_transport_clean) == 10000, "Unexpected row count."

trip_ids = df_transport_clean["trip_id_clean"]
assert trip_ids.notna().all(), "Missing trip IDs."
assert trip_ids.is_unique, "Repeated trip IDs."

assert df_transport_clean["mandi_id_clean"].isin(
    df_master_clean["mandi_id"]
).all(), "Mandi IDs not found in master."

assert df_transport_clean["destination_warehouse_clean"].notna().all()

distance = df_transport_clean["distance_km"]
assert distance.notna().all(), "Missing cleaned distances."
assert np.isfinite(distance).all() and distance.gt(0).all()

# Date-only and missing values must not become full timestamps.
for column in ["departure_time", "arrival_time"]:
    has_time = df_transport_clean[f"{column}_status"].eq(
        "Date and time available"
    )
    assert df_transport_clean[f"{column}_clean"].notna().equals(
        has_time
    ), f"Timestamp availability mismatch: {column}"

duration = df_transport_clean["transit_hours_clean"]
source = df_transport_clean["transit_duration_source"]

assert source.isin([
    "Calculated from timestamps",
    "Reported: timestamps incomplete",
    "Unavailable or invalid"
]).all()

assert duration.notna().equals(
    source.ne("Unavailable or invalid")
), "Duration availability mismatch."

assert np.isfinite(duration.dropna()).all()
assert duration.dropna().gt(0).all(), "Nonpositive cleaned durations."

for label, reference in [
    ("Calculated from timestamps", "transit_hours_calculated"),
    ("Reported: timestamps incomplete", "transit_hours_reported")
]:
    mask = source.eq(label)
    assert duration.loc[mask].eq(
        df_transport_clean.loc[mask, reference]
    ).all(), "Duration does not match its recorded source."

print("Transport validation passed.")
print("Total trips:", len(df_transport_clean))
print("Usable durations:", duration.notna().sum())
print("Unavailable durations:", duration.isna().sum())
print("Missing vehicle numbers:",
      df_transport_clean["vehicle_no_clean"].isna().sum())
print("Missing driver IDs:",
      df_transport_clean["driver_id_clean"].isna().sum())

Transport validation passed.
Total trips: 10000
Usable durations: 9715
Unavailable durations: 285
Missing vehicle numbers: 1560
Missing driver IDs: 1512


## Exporting file

In [25]:
from pathlib import Path

# Final column name → working column.
export_columns = {
    "trip_id": "trip_id_clean",
    "mandi_id": "mandi_id_clean",
    "destination_warehouse": "destination_warehouse_clean",
    "departure_time": "departure_time_clean",
    "arrival_time": "arrival_time_clean",
    "departure_date": "departure_time_date",
    "arrival_date": "arrival_time_date",
    "distance_km": "distance_km",
    "transit_hours": "transit_hours_clean",
    "vehicle_no": "vehicle_no_clean",
    "driver_id": "driver_id_clean",
    "mandi_id_status": "mandi_id_status",
    "distance_status": "distance_status",
    "departure_time_status": "departure_time_status",
    "arrival_time_status": "arrival_time_status",
    "departure_date_assumed": "departure_time_date_assumed",
    "arrival_date_assumed": "arrival_time_date_assumed",
    "transit_duration_source": "transit_duration_source",
    "transit_hours_calculated": "transit_hours_calculated",
    "transit_hours_reported": "transit_hours_reported",
    "reported_transit_negative": "reported_transit_negative",
    "transit_difference_hours": "transit_difference_hours"
}

transport_export = df_transport_clean[
    list(export_columns.values())
].copy()

transport_export.columns = list(export_columns.keys())

# Preserve the original fields for review.
original_columns = [
    "trip_id", "mandi_id", "destination_warehouse",
    "departure_time", "arrival_time", "transit_hours",
    "distance", "distance_unit", "vehicle_no", "driver_id"
]

for column in original_columns:
    transport_export[f"source_{column}"] = df_transport_clean[column]

# Keep calendar dates separate from full timestamps.
for column in ["departure_date", "arrival_date"]:
    transport_export[column] = (
        transport_export[column].dt.strftime("%Y-%m-%d")
    )

output_folder = Path(
    "/Users/rohankhanna/Documents/datathon/cleaned_data"
)
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "transport_logistics_cleaned.csv"

transport_export.to_csv(
    output_file,
    index=False,
    na_rep="NA",
    date_format="%Y-%m-%d %H:%M:%S",
    encoding="utf-8-sig"
)

# Check the exported file.
saved = pd.read_csv(output_file, keep_default_na=False)

assert saved.shape == transport_export.shape
assert saved["trip_id"].is_unique

print("Saved and checked:", output_file)
print("Rows:", len(saved))

Saved and checked: /Users/rohankhanna/Documents/datathon/cleaned_data/transport_logistics_cleaned.csv
Rows: 10000
